# Fourier Analysis - Error Bar

## Configuração
Edite apenas esta célula para adicionar/remover partículas ou frequências.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
np.seterr('warn')

from aux_functions import make_fitting_func, fit_curve, shorten_points, sort_experiments_by_freq
from treament_tcspc_data import extract_data_info_from_path, fetch_data_filenames, get_fund_freq_and_amp 

In [2]:
filenames_all = fetch_data_filenames("./sandbox")

In [3]:
exc_data_npy = [extract_data_info_from_path(data_path_npy) for data_path_npy in filenames_all if ("step" in data_path_npy) and (".npy" in data_path_npy) and ("exc_laser" in data_path_npy)]
data_txt = [extract_data_info_from_path(data_path_txt) for data_path_txt in filenames_all if ("step" in data_path_txt) and (".txt" in data_path_txt) and ("p10_" in data_path_txt)]
data_npy = [extract_data_info_from_path(data_path_npy) for data_path_npy in filenames_all if ("step" in data_path_npy) and (".npy" in data_path_npy) and ("p10_" in data_path_npy)]

In [4]:
freq_list = [1, 10, 100, 1000, 10000]
filtered_data = [d for d in data_npy if d["freq"] in freq_list]
filtered_exc_data = [d for d in exc_data_npy if d["freq"] in freq_list]
exc_data = sort_experiments_by_freq(filtered_exc_data)

## Plots

In [ ]:
## Parei aqui

lum_fund_freq_info = []

for idx, dic in enumerate(filtered_data):
    # if idx != 3:
    #     continue
    lum_fund_freq_info.append({
        "freq": dic["freq"],
        "fft_data": []
    })

    for rep_idx, rep in enumerate(dic["data"]):
        xs_lum, ys_lum = shorten_points(
            np.column_stack((rep[:,0], rep[:,1]))
            )
        
        lum_fund_freq_info[-1]["fft_data"].append(
            get_fund_freq_and_amp(xs_lum, ys_lum)
            )
        
    lum_fund_freq_info

idx = 3
p_arr = np.array([d["phase"] for d in lum_fund_freq_info[idx]["fft_data"]])
fund_freq_arr = np.array([d["fund_freq"] for d in lum_fund_freq_info[idx]["fft_data"]])

lum_fund_freq_info[idx]["freq"] , p_arr.mean(), p_arr.std(), fund_freq_arr.mean(), fund_freq_arr.std()
    


(1000,
 np.float64(-1.7467380388518055),
 np.float64(0.01955504954265341),
 np.float64(1058.6421591006833),
 np.float64(0.0))

In [ ]:
if False:
    xs_laser, ys_laser = shorten_points(
        np.column_stack((np.array(exc_data[idx]["data"])[0,:,0], np.array(exc_data[idx]["data"])[:,:,1].mean(axis=0)))
        )
    xs_lum, ys_lum = shorten_points(
        np.column_stack((np.array(filtered_data[idx]["data"])[0,:,0], np.array(filtered_data[idx]["data"])[:,:,1].mean(axis=0)))
        )
    # plot_time_and_freq_domain(xs_lum, ys_lum, ys_laser)    
    fund_freq_info.append({
        "freq": freq,
        "laser_fund_freq_dict": get_fund_freq_and_amp(xs_laser, ys_laser),
        "lum_fund_freq_dict": get_fund_freq_and_amp(xs_lum, ys_lum),
        })
    
if False:
    fig, ax1 = plt.subplots()
    ax1.plot(xs_laser*(1e-6), ys_laser/np.max(ys_laser), color="tab:red", label="laser")
    ax1.set_xlabel("Time ($\mu$s)")
    ax1.set_ylabel("Intensity Laser (a.u.)")
    ax1.legend(loc="upper left")
    ax2 = ax1.twinx()
    ax2.plot(xs_lum*(1e-6), ys_lum/np.max(ys_lum), color="tab:green", label="lum")
    ax2.set_ylabel("Intensity Luminescence (a.u.)")
    ax2.legend()
    plt.title(f"Time Domain - {freq} Hz")
    plt.show()
#-------

In [ ]:
laser_fund_freq_info = []

for idx, dic in enumerate(exc_data):
    # if idx != 3:
    #     continue
    laser_fund_freq_info.append({
        "freq": dic["freq"],
        "fft_data": []
    })

    for rep_idx, rep in enumerate(dic["data"]):
        xs_laser, ys_laser = shorten_points(
            np.column_stack((rep[:,0], rep[:,1]))
            )
        
        laser_fund_freq_info[-1]["fft_data"].append(
            get_fund_freq_and_amp(xs_laser, ys_laser)
            )

In [ ]:
idx = 0
p_arr = np.array([d["phase"] for d in laser_fund_freq_info[idx]["fft_data"]])
fund_freq_arr = np.array([d["fund_freq"] for d in laser_fund_freq_info[idx]["fft_data"]])

laser_fund_freq_info[idx]["freq"] , p_arr.mean(), p_arr.std(), fund_freq_arr.mean(), fund_freq_arr.std()

In [ ]:
pd_via_fft = []
for freq_dic in laser_fund_freq_info:
    lum_p = freq_dic["lum_fund_freq_dict"]["phase"]
    laser_p = freq_dic["laser_fund_freq_dict"]["phase"]
    pd_via_fft.append((freq_dic['freq'], np.degrees(-lum_p + laser_p)))

for idx, el in enumerate(pd_via_fft):
    # color = colors[idx % len(colors)]
    freq, phase = el
    plt.errorbar(
        freq,
        phase,
        # yerr=1.0,  # error_bars[idx]["mean_std"],
        label=f"{freq} Hz - FFT",
        fmt="x",
        capsize=5,
        # color=color,
    )

plt.xscale("log")
plt.title("Phase Difference Comparison")
plt.ylabel("Degrees (º)")
plt.xlabel("f (Hz)")
plt.legend()
plt.show()